# DrishtiYOLO — Training Notebook
YOLO11s fine-tuned on IDD Detection (Indian Driving Dataset)

## Run order — every session (fresh or resumed)

| Cell | What it does | Skip condition |
|------|-------------|----------------|
| **1** | GPU check + install | Never skip — always run first |
| **2** | Mount Google Drive | Never skip |
| **3** | Download + extract IDD (~22 GB) | Skips if already extracted this session |
| **4** | Convert VOC XML → YOLO format | Skips if `idd_yolo/` already has 41 k images |
| **5** | Delete archive to free ~50 GB | Skips if files already gone |
| **6** | Write dataset YAML | Always safe to rerun |
| **7** | **Train YOLO11s** | Auto-resumes if `last.pt` found on Drive; otherwise starts fresh |

**Cell 9** (bottom) = optional smoke test — only needed once to validate the pipeline.

In [ ]:
# ── Cell 1 — Environment check + install ──────────────────────
# Restart runtime if nvidia-smi shows K80 — request a new one.
!nvidia-smi

!pip install -q ultralytics wandb

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU : {p.name}")
    print(f"VRAM: {p.total_memory / 1e9:.1f} GB")

Sat May 16 15:57:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── Cell 2 — Mount Google Drive ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/drishti_runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Checkpoint dir: {DRIVE_ROOT}")

Mounted at /content/drive
Checkpoint dir: /content/drive/MyDrive/drishti_runs


In [ ]:
# ── Cell 3 — Download + Extract IDD Detection ─────────────────
# Idempotent: skips download if archive already present and valid.
#             skips extraction if /content/idd_detection already exists.

import zipfile, tarfile
from pathlib import Path

# ── Configuration ── paste your IDD token URL below ──────────────
IDD_TOKEN_URL = "PASTE_YOUR_TOKEN_URL_HERE"   # <── paste here
COOKIES_FILE  = "idd.insaan.iiit.ac.in_cookies.txt"
# Set True to upload the archive from your local machine instead of wget
UPLOAD_FROM_LOCAL = False
# ─────────────────────────────────────────────────────────────────

IDD_ZIP     = Path('/content/idd_detection.zip')
IDD_EXTRACT = Path('/content/idd_detection')
MIN_VALID_BYTES = 100 * 1024 * 1024   # anything < 100 MB is an HTML error page

def _stale(p: Path) -> bool:
    return p.exists() and p.stat().st_size < MIN_VALID_BYTES

if _stale(IDD_ZIP):
    print(f"Removing stale file ({IDD_ZIP.stat().st_size / 1024:.1f} KB — was an HTML error page)")
    IDD_ZIP.unlink()

def check_cookie_file(path: str):
    p = Path(path)
    if not p.exists():
        return False, "file not found"
    content = p.read_text()
    if 'sessionid' not in content and 'csrftoken' not in content:
        return False, "no sessionid/csrftoken — export cookies while logged in to idd.insaan.iiit.ac.in"
    return True, f"valid ({p.stat().st_size} bytes)"

def detect_and_extract(src: Path, dst: Path):
    with open(src, 'rb') as f:
        magic = f.read(6)
    if magic[:2] == b'PK':
        fmt = 'zip'
    elif magic[:2] == b'\x1f\x8b':
        fmt = 'tar.gz'
    elif magic[:3] == b'BZh':
        fmt = 'tar.bz2'
    else:
        with open(src, 'rb') as f:
            f.seek(257); tar_magic = f.read(5)
        fmt = 'tar' if tar_magic == b'ustar' else 'unknown'
    print(f"Archive format detected: {fmt}")
    dst.mkdir(parents=True, exist_ok=True)
    if fmt == 'zip':
        with zipfile.ZipFile(src) as zf:
            zf.extractall(dst)
    elif fmt == 'tar.gz':
        with tarfile.open(src, 'r:gz') as tf:
            tf.extractall(dst, filter='data')
    elif fmt == 'tar.bz2':
        with tarfile.open(src, 'r:bz2') as tf:
            tf.extractall(dst, filter='data')
    elif fmt == 'tar':
        with tarfile.open(src, 'r:') as tf:
            tf.extractall(dst, filter='data')
    else:
        raise RuntimeError(
            f"Unknown archive format (magic={magic!r}). Run: !file {src}"
        )

# ── Step 1: Download ──────────────────────────────────────────────
if UPLOAD_FROM_LOCAL:
    from google.colab import files
    print("Upload the IDD Detection archive from your local machine:")
    up = files.upload()
    Path(list(up.keys())[0]).rename(IDD_ZIP)
    print(f"Saved → {IDD_ZIP}")

elif IDD_ZIP.exists():
    print(f"Archive already present ({IDD_ZIP.stat().st_size / 1e9:.2f} GB) — skipping download.")

else:
    if IDD_TOKEN_URL == "PASTE_YOUR_TOKEN_URL_HERE":
        raise ValueError("Paste your IDD download token URL into IDD_TOKEN_URL above.")

    cookie_ok, cookie_msg = check_cookie_file(COOKIES_FILE)
    if not cookie_ok:
        print(f"Cookie file: {cookie_msg}")
        print(f"Please upload '{COOKIES_FILE}' (exported from 'Get cookies.txt LOCALLY' extension):")
        from google.colab import files
        files.upload()
        cookie_ok, cookie_msg = check_cookie_file(COOKIES_FILE)
        if not cookie_ok:
            raise RuntimeError(f"Cookie file still invalid: {cookie_msg}")

    print(f"Cookie: {cookie_msg}")
    print("Downloading IDD Detection (~22.8 GB) — expect 10–25 min on Colab...")
    !wget --load-cookies "{COOKIES_FILE}" --keep-session-cookies \
          -q --show-progress "{IDD_TOKEN_URL}" -O {IDD_ZIP}

    size = IDD_ZIP.stat().st_size if IDD_ZIP.exists() else 0
    if size < MIN_VALID_BYTES:
        IDD_ZIP.unlink(missing_ok=True)
        raise RuntimeError(
            f"Download failed — got {size/1024:.1f} KB (expected ~22 GB).\n"
            "Token URL may have expired — generate a fresh one from the IDD portal."
        )
    print(f"Download complete — {size / 1e9:.2f} GB")

# ── Step 2: Extract ───────────────────────────────────────────────
if IDD_EXTRACT.exists():
    n = sum(1 for _ in IDD_EXTRACT.rglob('*'))
    print(f"Already extracted ({n:,} files in {IDD_EXTRACT}) — skipping extraction.")
else:
    print("Extracting archive (may take a few minutes)...")
    detect_and_extract(IDD_ZIP, IDD_EXTRACT)
    print("Extraction complete.")

In [ ]:
# ── Cell 4 — Convert IDD VOC XML → YOLO format ────────────────
# Idempotent: if idd_yolo/ already has all 41,794 images, prints a
#             summary and exits — safe to rerun any number of times.
# Self-contained: finds the IDD root automatically (no Cell 4 inspect needed).

import shutil, xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm

IDD_EXTRACT = Path('/content/idd_detection')
YOLO_DIR    = Path('/content/idd_yolo')

EXPECTED_TRAIN = 31569
EXPECTED_VAL   = 10225

# ── Skip guard ────────────────────────────────────────────────────
def _count(split):
    d = YOLO_DIR / 'images' / split
    return len(list(d.glob('*'))) if d.exists() else 0

if _count('train') >= EXPECTED_TRAIN and _count('val') >= EXPECTED_VAL:
    print(f"Conversion already complete:")
    print(f"  train images : {_count('train'):,}")
    print(f"  val   images : {_count('val'):,}")
    print("Nothing to do — skipping conversion.")
else:
    # ── Find IDD root (handles sub-folder layout) ─────────────────
    IDD_ROOT = None
    candidates = [IDD_EXTRACT] + (list(IDD_EXTRACT.iterdir()) if IDD_EXTRACT.exists() else [])
    for c in candidates:
        if c.is_dir() and (c / 'Annotations').exists() and (c / 'JPEGImages').exists():
            IDD_ROOT = c
            break
    if IDD_ROOT is None:
        raise RuntimeError(
            "Cannot find IDD dataset at /content/idd_detection.\n"
            "Run Cell 3 first to download and extract the archive."
        )

    ANN_DIR   = IDD_ROOT / 'Annotations'
    IMG_DIR   = IDD_ROOT / 'JPEGImages'
    TRAIN_TXT = IDD_ROOT / 'train.txt'
    VAL_TXT   = IDD_ROOT / 'val.txt'
    print(f"IDD root : {IDD_ROOT}")
    print(f"XML files: {len(list(ANN_DIR.rglob('*.xml'))):,}")

    # ── Class map ─────────────────────────────────────────────────
    IDD_CLASSES = [
        'auto_rickshaw', 'bicycle', 'bus', 'car', 'caravan',
        'electric_vehicle', 'motorcycle', 'person', 'rider',
        'traffic_light', 'traffic_sign', 'truck',
        'vehicle_fallback', 'animal', 'wheelchair',
    ]
    CLASS_TO_IDX = {c: i for i, c in enumerate(IDD_CLASSES)}

    ALIAS = {
        'autorickshaw':     'auto_rickshaw',
        'auto rickshaw':    'auto_rickshaw',
        'electricvehicle':  'electric_vehicle',
        'electric vehicle': 'electric_vehicle',
        'trafficlight':     'traffic_light',
        'traffic light':    'traffic_light',
        'trafficsign':      'traffic_sign',
        'traffic sign':     'traffic_sign',
        'vehiclefallback':  'vehicle_fallback',
        'vehicle fallback': 'vehicle_fallback',
        'wheelchar':        'wheelchair',
    }

    def normalise(name):
        n = name.strip().lower().replace('-', '_')
        return ALIAS.get(n, n)

    def stem_to_unique(stem):
        # e.g. 'frontFar/SEQ/000123_r' → 'frontFar__SEQ__000123_r'
        return stem.replace('/', '__')

    def find_image(stem):
        for ext in ('.jpg', '.jpeg', '.png'):
            p = IMG_DIR / (stem + ext)
            if p.exists():
                return p
        return None

    def find_xml(stem):
        p = ANN_DIR / (stem + '.xml')
        return p if p.exists() else None

    def convert_xml(xml_path):
        root = ET.parse(xml_path).getroot()
        size = root.find('size')
        if size is None:
            return []
        w = int(size.findtext('width', 0))
        h = int(size.findtext('height', 0))
        if w == 0 or h == 0:
            return []
        lines = []
        for obj in root.findall('object'):
            cls = CLASS_TO_IDX.get(normalise(obj.findtext('name', '')))
            if cls is None:
                continue
            b = obj.find('bndbox')
            if b is None:
                continue
            xmin = max(0.0, float(b.findtext('xmin', 0)))
            ymin = max(0.0, float(b.findtext('ymin', 0)))
            xmax = min(float(w), float(b.findtext('xmax', 0)))
            ymax = min(float(h), float(b.findtext('ymax', 0)))
            if xmax <= xmin or ymax <= ymin:
                continue
            cx = (xmin + xmax) / 2 / w
            cy = (ymin + ymax) / 2 / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h
            lines.append(f'{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        return lines

    def convert_split(split, txt_path):
        stems = [l.strip() for l in open(txt_path) if l.strip()]
        out_img = YOLO_DIR / 'images' / split
        out_lbl = YOLO_DIR / 'labels' / split
        out_img.mkdir(parents=True, exist_ok=True)
        out_lbl.mkdir(parents=True, exist_ok=True)
        converted = skipped = missing_img = missing_xml = 0
        for stem in tqdm(stems, desc=split):
            img = find_image(stem)
            xml = find_xml(stem)
            if img is None:
                missing_img += 1; continue
            if xml is None:
                missing_xml += 1; continue
            unique  = stem_to_unique(stem)
            dst_img = out_img / (unique + img.suffix)
            if dst_img.exists():
                skipped += 1; continue
            shutil.copy2(img, dst_img)
            with open(out_lbl / (unique + '.txt'), 'w') as f:
                f.write('\n'.join(convert_xml(xml)))
            converted += 1
        print(f"  Converted : {converted:,}  |  Already existed: {skipped:,}")
        print(f"  Missing img: {missing_img}  |  Missing xml: {missing_xml}")
        print(f"  Total → images: {len(list(out_img.glob('*'))):,}   "
              f"labels: {len(list(out_lbl.glob('*'))):,}")

    for split, txt in [('train', TRAIN_TXT), ('val', VAL_TXT)]:
        print(f"\n[{split}]")
        convert_split(split, txt)

    print('\nConversion complete.')

In [ ]:
# ── Cell 5 — Free disk space (delete archive + raw IDD) ───────
# Idempotent: safe to rerun — only deletes files that still exist.
# Guard: refuses to delete if conversion is not complete, so you
#        cannot accidentally lose the source data before idd_yolo/ is ready.

import shutil
from pathlib import Path

YOLO_DIR       = Path('/content/idd_yolo')
IDD_ZIP        = Path('/content/idd_detection.zip')
IDD_EXTRACT    = Path('/content/idd_detection')
EXPECTED_TRAIN = 31569
EXPECTED_VAL   = 10225

def _count(split):
    d = YOLO_DIR / 'images' / split
    return len(list(d.glob('*'))) if d.exists() else 0

train_ok = _count('train') >= EXPECTED_TRAIN
val_ok   = _count('val')   >= EXPECTED_VAL

if not (train_ok and val_ok):
    print("ERROR: Conversion not complete — run Cell 4 first.")
    print(f"  train images found: {_count('train'):,}  (need {EXPECTED_TRAIN:,})")
    print(f"  val   images found: {_count('val'):,}  (need {EXPECTED_VAL:,})")
else:
    freed = 0
    for path in [IDD_EXTRACT, IDD_ZIP]:
        if path.exists():
            if path.is_dir():
                size = sum(f.stat().st_size for f in path.rglob('*') if f.is_file())
                shutil.rmtree(path)
            else:
                size = path.stat().st_size
                path.unlink()
            freed += size
            print(f"Deleted {path}  ({size / 1e9:.1f} GB freed)")
        else:
            print(f"Already gone: {path}")

    total, used, free = shutil.disk_usage('/content')
    print(f"\nDisk: {free / 1e9:.1f} GB free  |  {used / 1e9:.1f} GB used  |  {total / 1e9:.1f} GB total")
    if freed:
        print(f"Total freed this run: {freed / 1e9:.1f} GB")

In [10]:
# ── Cell 6 — Write dataset YAML ───────────────────────────────
import yaml
from pathlib import Path

YOLO_DIR = Path('/content/idd_yolo')

dataset_cfg = {
    'path': str(YOLO_DIR),
    'train': 'images/train',
    'val':   'images/val',
    'nc': 15,
    'names': [
        'auto_rickshaw', 'bicycle', 'bus', 'car', 'caravan',
        'electric_vehicle', 'motorcycle', 'person', 'rider',
        'traffic_light', 'traffic_sign', 'truck',
        'vehicle_fallback', 'animal', 'wheelchair'
    ]
}

yaml_path = Path('/content/idd.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False)

print(f'Dataset YAML written to {yaml_path}')
!cat {yaml_path}

# Sanity check — count images and labels
for split in ['train', 'val']:
    n_img = len(list((YOLO_DIR / 'images' / split).glob('*')))
    n_lbl = len(list((YOLO_DIR / 'labels' / split).glob('*')))
    print(f"{split:5s}  images: {n_img:6d}  labels: {n_lbl:6d}")

Dataset YAML written to /content/idd.yaml
names:
- auto_rickshaw
- bicycle
- bus
- car
- caravan
- electric_vehicle
- motorcycle
- person
- rider
- traffic_light
- traffic_sign
- truck
- vehicle_fallback
- animal
- wheelchair
nc: 15
path: /content/idd_yolo
train: images/train
val: images/val
train  images:  31569  labels:  31569
val    images:  10225  labels:  10225


In [ ]:
# ── Cell 7 — Train YOLO11s (auto-resume if interrupted) ────────
# • No checkpoint on Drive  → starts fresh from COCO-pretrained weights
# • last.pt found on Drive  → resumes from where the previous session stopped
#
# Expected time: ~27 min/epoch × 50 epochs ≈ 22 hr total (2 Colab sessions).
# Just run this cell every new session — it figures out what to do automatically.
#
# Optional W&B logging (free tier):
#   import wandb; wandb.login()   # paste API key when prompted

from pathlib import Path
from ultralytics import YOLO

DRIVE_ROOT = '/content/drive/MyDrive/drishti_runs'
yaml_path  = Path('/content/idd.yaml')
LAST_PT    = Path(DRIVE_ROOT) / 'yolo11s_idd_v1' / 'weights' / 'last.pt'
BEST_PT    = Path(DRIVE_ROOT) / 'yolo11s_idd_v1' / 'weights' / 'best.pt'

if LAST_PT.exists():
    print(f"Checkpoint found: {LAST_PT}")
    print("Resuming from last checkpoint...")
    model   = YOLO(str(LAST_PT))
    results = model.train(resume=True)
else:
    print("No checkpoint found — starting fresh training from COCO weights.")
    model   = YOLO('yolo11s.pt')
    results = model.train(
        data        = str(yaml_path),
        imgsz       = 640,
        epochs      = 50,          # ~27 min/epoch × 50 ≈ 22 hr = 2 sessions
        batch       = 16,          # reduce to 8 if OOM
        optimizer   = 'AdamW',
        lr0         = 0.001,
        amp         = True,
        cache       = False,       # disk cache needs 270 GB — not available on Colab
        patience    = 15,
        save_period = 5,           # checkpoint every 5 epochs to Drive
        project     = DRIVE_ROOT,
        name        = 'yolo11s_idd_v1',
        exist_ok    = True,
    )

# Backup best weights with a timestamped name so they survive future runs
import shutil, datetime
if BEST_PT.exists():
    ts      = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    bak     = Path(DRIVE_ROOT) / f'best_yolo11s_idd_v1_{ts}.pt'
    shutil.copy2(BEST_PT, bak)
    print(f'best.pt backed up → {bak}')

In [ ]:
# ── Cell 9 — YOLO11n smoke test (run on Day 1 before full run) ─
# Verifies the full pipeline in ~1 hr instead of 12+.
# Check mAP output before committing to the full YOLO11s run.

from pathlib import Path
from ultralytics import YOLO

DRIVE_ROOT = '/content/drive/MyDrive/drishti_runs'
yaml_path  = Path('/content/idd.yaml')

nano = YOLO('yolo11n.pt')
nano.train(
    data        = str(yaml_path),
    imgsz       = 640,
    epochs      = 10,
    batch       = 16,
    cache       = 'disk',
    project     = DRIVE_ROOT,
    name        = 'yolo11n_idd_smoke',
    exist_ok    = True,
)
print('Smoke test done — inspect mAP before launching Cell 7.')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/idd.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8